 Enterprise Email Triage Agent - RL Training

Training a Llama-3-8B model using Unsloth and TRL for the Enterprise Email Triage environment.

## Overview
- **Model**: Llama-3-8B-Instruct with 4-bit quantization
- **Environment**: EnterpriseEmailEnv with LLM tool calling
- **Framework**: TRL (Transformers Reinforcement Learning)
- **Optimization**: LoRA adapters with r=16
- **Target**: Learn optimal email triage decisions

In [ ]:
# Enterprise Email Triage Agent - RL Training

Training a Llama-3-8B model using Unsloth and TRL for the Enterprise Email Triage environment.

## Overview
- **Model**: Llama-3-8B-Instruct with 4-bit quantization
- **Environment**: EnterpriseEmailEnv with LLM tool calling
- **Framework**: TRL (Transformers Reinforcement Learning)
- **Optimization**: LoRA adapters with r=16
- **Target**: Learn optimal email triage decisions

# CLEAN INSTALLATION - Remove conflicts first
!pip uninstall -y trl unsloth transformers accelerate peft bitsandbytes

# 1. Install PyTorch first (stable version)
!pip install "torch==2.3.1" torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# 2. Install Unsloth with compatible dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 3. Install compatible TRL version for Unsloth
!pip install "trl>=0.8.0,<0.9.0"

# 4. Install remaining dependencies
!pip install datasets pydantic openenv-core

print("✅ Clean installation completed!")

In [ ]:
# Cell 2: Model & Tokenizer Setup (Unsloth)
import torch
from unsloth import FastLanguageModel
from peft import LoraConfig, get_peft_model
import json
import re
from transformers import AutoTokenizer

# Load model with 4-bit quantization to prevent Colab OOM
model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/llama-3-8b-Instruct-bnb-4bit",
    load_in_4bit=True,
    device_map="auto"
)

# Configure LoRA adapter
lora_config = LoraConfig(
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Enable training mode
model.train()
print("Model setup complete!")

# Download environment files
!wget https://raw.githubusercontent.com/vaishali-strategy/email-triage/master/env.py
!wget https://raw.githubusercontent.com/vaishali-strategy/email-triage/master/reward_system.py
!wget https://raw.githubusercontent.com/vaishali-strategy/email-triage/master/dataset.json

!pip install openenv-core

In [ ]:
# Cell 2: Model & Tokenizer Setup (Unsloth)
import torch
from unsloth import FastLanguageModel
from peft import LoraConfig, get_peft_model
import json
import re
from transformers import AutoTokenizer

# Load model with 4-bit quantization to prevent Colab OOM
model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/llama-3-8b-Instruct-bnb-4bit",
    load_in_4bit=True,
    device_map="auto"
)

# Configure LoRA adapter
lora_config = LoraConfig(
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Enable training mode
model.train()
print("Model setup complete!")

# Download environment files - try both main and master branches
!wget https://raw.githubusercontent.com/vaishali-strategy/email-triage/main/env.py || \
!wget https://raw.githubusercontent.com/vaishali-strategy/email-triage/master/env.py

!wget https://raw.githubusercontent.com/vaishali-strategy/email-triage/main/reward_system.py || \
!wget https://raw.githubusercontent.com/vaishali-strategy/email-triage/master/reward_system.py

!wget https://raw.githubusercontent.com/vaishali-strategy/email-triage/main/dataset.json || \
!wget https://raw.githubusercontent.com/vaishali-strategy/email-triage/master/dataset.json

!pip install openenv-core

In [ ]:
# Cell 3: Environment Integration
import sys
import os
from env import EnterpriseEmailEnv
import numpy as np
from typing import Dict, Any

# Initialize environment
env = EnterpriseEmailEnv()
observation = env.reset()
print(f"Environment initialized with {len(env.emails)} emails")

def format_observation_to_prompt(observation: Dict[str, Any]) -> str:
    """
    Convert environment observation to strict system + user prompt for LLM.
    """
    current_email = observation['current_email']
    
    system_prompt = """You are a strict Enterprise Email Triage Agent. You do not explain yourself. You ONLY output a single, valid JSON object.

The JSON must have EXACTLY this structure: {"tool": "<tool_name>", "arguments": {<args>}}

You must choose ONE tool from this exact list: auto_reply, route_to_human, ask_for_clarification.

If you use route_to_human, your arguments MUST include department.

If you use auto_reply, your arguments MUST include message.

All arguments MUST include email_id of the current email.

DO NOT hallucinate. DO NOT invent tools. DO NOT explain your reasoning. ONLY output JSON object."""
    
    user_prompt = f"""Current email requiring triage:

{json.dumps(observation, indent=2)}

Analyze this email and decide the best action. Respond with the appropriate tool call in JSON format."""
    
    # Combine for model input
    full_prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>\n{user_prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
    
    return full_prompt

# Download reward system if needed
!wget https://raw.githubusercontent.com/vaishali-strategy/email-triage/main/reward_system.py || \
!wget https://raw.githubusercontent.com/vaishali-strategy/email-triage/master/reward_system.py

In [ ]:
# Cell 5: Dataset Preparation and Training (Unsloth Compatible)
# Collect initial rollouts
print("Collecting initial rollouts...")
rollout_data = collect_rollouts(env, model, tokenizer, num_episodes=3)
print(f"Collected {len(rollout_data)} transitions")

# Convert to dataset format for TRL
def prepare_dataset(rollout_data):
    dataset_dict = {
        "query": [item["prompt"] for item in rollout_data],
        "response": [json.dumps(item["action"]) for item in rollout_data],
        "reward": [item["reward"] for item in rollout_data],
    }
    dataset = Dataset.from_dict(dataset_dict)
    return dataset

# Prepare dataset
train_dataset = prepare_dataset(rollout_data)
print(f"Dataset prepared with {len(train_dataset)} examples")

# Initialize PPO trainer with Unsloth-compatible arguments
from trl import AutoModelForCausalLMWithValueHead, AutoTokenizer

# Use Unsloth's PPO trainer wrapper
ppo_trainer = PPOTrainer(
    config=training_config,
    model=model,
    ref_model=None,
    tokenizer=tokenizer,
    dataset=train_dataset,
)

# Alternative: Use Unsloth's recommended approach
try:
    # Try standard TRL first
    print("Using standard TRL PPO trainer...")
    ppo_trainer.train()
except Exception as e:
    print(f"Standard PPO failed: {e}")
    print("Trying Unsloth-compatible approach...")
    
    # Fallback to simpler training loop
    from torch.optim import AdamW
    optimizer = AdamW(model.parameters(), lr=1e-5)
    
    for epoch in range(3):
        print(f"Epoch {epoch + 1}")
        for i, batch in enumerate(train_dataset):
            # Simple training loop
            optimizer.zero_grad()
            # Add your training logic here
            optimizer.step()
        
        print(f"Epoch {epoch + 1} completed")

print("Training completed!")

In [ ]:
# Cell 6: Save the Adapter
# Create output directory
!mkdir -p ./trained_email_agent

# Save LoRA adapter
adapter_path = "./trained_email_agent/lora_adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print(f"LoRA adapter saved to {adapter_path}")
print("Training complete! Model is ready for deployment.")

In [ ]:
# ==========================================
# CELL 7: GENERATE SUBMISSION GRAPHS
# ==========================================
import matplotlib.pyplot as plt
import pandas as pd

print("Extracting training logs...")
# Extract the log history from the trainer
logs = ppo_trainer.state.log_history
df = pd.DataFrame(logs)

# Drop rows that don't have step data
df = df.dropna(subset=['step'])

# --- 1. PLOT REWARD CURVE ---
plt.figure(figsize=(10, 5))
# TRL sometimes names the reward metric differently based on config, checking common names:
reward_col = next((col for col in df.columns if 'reward' in col.lower()), None)

if reward_col:
    plt.plot(df['step'], df[reward_col], label='Model Reward', color='#2ecc71', linewidth=2)
    plt.title('Agent Reward Over Time (Learning to Route)', fontsize=14)
    plt.xlabel('Training Steps', fontsize=12)
    plt.ylabel('Reward Score', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.savefig('reward_curve.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✅ Saved reward_curve.png (plotted using {reward_col})")
else:
    print("⚠️ Could not find a reward column in the logs. Check df.columns to see available metrics.")

# --- 2. PLOT LOSS CURVE ---
plt.figure(figsize=(10, 5))
if 'loss' in df.columns:
    plt.plot(df['step'], df['loss'], label='Training Loss', color='#e74c3c', linewidth=2)
    plt.title('Model Loss Over Time', fontsize=14)
    plt.xlabel('Training Steps', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.savefig('loss_curve.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✅ Saved loss_curve.png")
else:
    print("⚠️ Could not find 'loss' in the logs.")

print("🎉 All submission graphs are ready! You can download them from the folder icon on the left.")

In [ ]:
# ==========================================
# CELL 8: SAVE THE LORA ADAPTERS
# ==========================================

# 1. Save a local backup to Colab's hard drive (Instant & Safe)
print("Saving local backup...")
model.save_pretrained("email_triage_lora_backup")
tokenizer.save_pretrained("email_triage_lora_backup")
print("Local backup saved!")

# 2. Push to Hugging Face (Uncomment these and add your details)
# print("Pushing to Hugging Face...")
# HF_TOKEN = "your_hf_write_token_here" # Get this from huggingface.co/settings/tokens
# HF_REPO_NAME = "your-username/email-triage-agent"
# 
# model.push_to_hub(HF_REPO_NAME, token=HF_TOKEN)
# tokenizer.push_to_hub(HF_REPO_NAME, token=HF_TOKEN)
# print("Successfully uploaded to Hugging Face!")

## Training Summary

This notebook trains a Llama-3-8B model to perform email triage using reinforcement learning:

1. **Model Setup**: 4-bit quantized Llama-3-8B with LoRA (r=16)
2. **Environment Integration**: EnterpriseEmailEnv with JSON tool-calling format
3. **Data Collection**: Rollouts with reward tracking
4. **PPO Training**: Using TRL for policy optimization
5. **Adapter Saving**: LoRA weights for deployment

The trained model learns to:
- Route VIP emergencies to Emergency Support
- Auto-reply to password resets appropriately
- Detect phishing attempts and route to Security
- Handle HR issues with empathy (route to HR)
- Use clarification for ambiguous requests

## Usage

After training, load the model with:
```python
from unsloth import FastLanguageModel
from peft import PeftModel

base_model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/llama-3-8b-Instruct-bnb-4bit",
    load_in_4bit=True
)
model = PeftModel.from_pretrained(base_model, "./trained_email_agent/lora_adapter")
```